In [1]:
import pandas as pd
import numpy as np
import random
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# Memuat dataset
# Pastikan file 'mental_health_analysis.csv' berada di direktori yang sama
df = pd.read_csv('mental_health_analysis.csv')

# Menampilkan 5 baris pertama untuk memastikan data termuat dengan benar
print("Data Awal:")
print(df.head())

Data Awal:
   User_ID  Age Gender  Social_Media_Hours  Exercise_Hours  Sleep_Hours  \
0        1   16      F            9.654486        2.458001     5.198926   
1        2   17      M            9.158143        0.392095     8.866097   
2        3   15      M            5.028755        0.520119     4.943095   
3        4   17      F            7.951103        1.022630     5.262773   
4        5   17      F            1.357459        1.225462     6.196080   

   Screen_Time_Hours  Survey_Stress_Score  Wearable_Stress_Score  \
0           8.158189                    3               0.288962   
1           5.151993                    5               0.409446   
2           9.209325                    2               0.423837   
3           9.823658                    5               0.666021   
4          11.338990                    5               0.928060   

  Support_System Academic_Performance  
0       Moderate            Excellent  
1       Moderate                 Good  
2       M

In [2]:
# Melakukan One-Hot Encoding pada variabel kategorikal
df_processed = pd.get_dummies(df, columns=['Gender', 'Support_System', 'Academic_Performance'], drop_first=True)

# Menghapus User_ID karena tidak relevan untuk prediksi
df_processed = df_processed.drop('User_ID', axis=1)

# Memisahkan fitur (X) dan target (y)
X = df_processed.drop('Survey_Stress_Score', axis=1)
y = df_processed['Survey_Stress_Score']

# Menyimpan nama-nama fitur untuk interpretasi nanti
feature_names = X.columns.tolist()

print("\nData Setelah Preprocessing (Contoh):")
print(X.head())
print(f"\nTotal fitur yang akan dianalisis: {len(feature_names)}")
print("Nama Fitur:", feature_names)


Data Setelah Preprocessing (Contoh):
   Age  Social_Media_Hours  Exercise_Hours  Sleep_Hours  Screen_Time_Hours  \
0   16            9.654486        2.458001     5.198926           8.158189   
1   17            9.158143        0.392095     8.866097           5.151993   
2   15            5.028755        0.520119     4.943095           9.209325   
3   17            7.951103        1.022630     5.262773           9.823658   
4   17            1.357459        1.225462     6.196080          11.338990   

   Wearable_Stress_Score  Gender_M  Support_System_Low  \
0               0.288962     False               False   
1               0.409446      True               False   
2               0.423837      True               False   
3               0.666021     False               False   
4               0.928060     False               False   

   Support_System_Moderate  Academic_Performance_Excellent  \
0                     True                            True   
1                   

In [3]:
# --- FUNGSI FITNESS ---
# Fungsi ini adalah inti dari AG kita.
# Ia menghitung Adjusted R-squared untuk sebuah kombinasi fitur (kromosom).
def calculate_fitness(chromosome, X, y):
    # Pilih kolom fitur berdasarkan gen '1' di kromosom
    selected_features_indices = [i for i, gene in enumerate(chromosome) if gene == 1]
    
    # Jika tidak ada fitur yang terpilih, fitness = 0
    if not selected_features_indices:
        return 0.0
        
    X_subset = X.iloc[:, selected_features_indices]
    
    # Membuat dan melatih model regresi linear
    model = LinearRegression()
    model.fit(X_subset, y)
    
    # Menghitung R-squared
    r2 = model.score(X_subset, y)
    
    # Menghitung Adjusted R-squared
    n = len(y)  # Jumlah sampel
    p = len(selected_features_indices) # Jumlah prediktor/fitur
    
    if n - p - 1 == 0: # Mencegah pembagian dengan nol
        return 0.0
        
    adj_r2 = 1 - (1 - r2) * (n - 1) / (n - p - 1)
    return adj_r2

# --- FUNGSI LAINNYA UNTUK AG ---
def create_individual(num_features):
    """Menciptakan satu kromosom acak."""
    return [random.randint(0, 1) for _ in range(num_features)]

def create_population(pop_size, num_features):
    """Menciptakan populasi awal."""
    return [create_individual(num_features) for _ in range(pop_size)]

def selection(population, fitnesses):
    """Memilih satu individu menggunakan tournament selection."""
    tournament_size = 3
    tournament_indices = random.sample(range(len(population)), tournament_size)
    tournament_fitnesses = [fitnesses[i] for i in tournament_indices]
    
    winner_index = tournament_indices[np.argmax(tournament_fitnesses)]
    return population[winner_index]

def crossover(parent1, parent2):
    """Melakukan single-point crossover."""
    crossover_point = random.randint(1, len(parent1) - 1)
    child1 = parent1[:crossover_point] + parent2[crossover_point:]
    child2 = parent2[:crossover_point] + parent1[crossover_point:]
    return child1, child2

def mutation(individual, mutation_rate):
    """Melakukan bit-flip mutation."""
    for i in range(len(individual)):
        if random.random() < mutation_rate:
            individual[i] = 1 - individual[i] # Flip the bit
    return individual

print("\nKomponen Algoritma Genetika telah didefinisikan.")


Komponen Algoritma Genetika telah didefinisikan.


In [4]:
# --- PARAMETER AG ---
POPULATION_SIZE = 50
NUM_GENERATIONS = 30
MUTATION_RATE = 0.02
NUM_FEATURES = len(feature_names)

# 1. Inisialisasi Populasi
population = create_population(POPULATION_SIZE, NUM_FEATURES)

best_overall_fitness = -1
best_overall_chromosome = []

# --- LOOP EVOLUSI ---
for gen in range(NUM_GENERATIONS):
    # 2. Evaluasi Fitness
    fitnesses = [calculate_fitness(ind, X, y) for ind in population]
    
    # Simpan individu terbaik dari generasi ini
    best_gen_fitness = max(fitnesses)
    best_gen_idx = np.argmax(fitnesses)
    
    if best_gen_fitness > best_overall_fitness:
        best_overall_fitness = best_gen_fitness
        best_overall_chromosome = population[best_gen_idx]

    print(f"Generasi {gen+1}/{NUM_GENERATIONS} | Fitness Terbaik: {best_overall_fitness:.4f}")

    # 3. Buat Generasi Baru
    new_population = []
    while len(new_population) < POPULATION_SIZE:
        # 3a. Seleksi
        parent1 = selection(population, fitnesses)
        parent2 = selection(population, fitnesses)
        
        # 3b. Crossover
        child1, child2 = crossover(parent1, parent2)
        
        # 3c. Mutasi
        mutated_child1 = mutation(child1, MUTATION_RATE)
        mutated_child2 = mutation(child2, MUTATION_RATE)
        
        new_population.extend([mutated_child1, mutated_child2])
        
    population = new_population[:POPULATION_SIZE]

Generasi 1/30 | Fitness Terbaik: 0.0002
Generasi 2/30 | Fitness Terbaik: 0.0004
Generasi 3/30 | Fitness Terbaik: 0.0004
Generasi 4/30 | Fitness Terbaik: 0.0005
Generasi 5/30 | Fitness Terbaik: 0.0005
Generasi 6/30 | Fitness Terbaik: 0.0005
Generasi 7/30 | Fitness Terbaik: 0.0005
Generasi 8/30 | Fitness Terbaik: 0.0005
Generasi 9/30 | Fitness Terbaik: 0.0005
Generasi 10/30 | Fitness Terbaik: 0.0005
Generasi 11/30 | Fitness Terbaik: 0.0005
Generasi 12/30 | Fitness Terbaik: 0.0005
Generasi 13/30 | Fitness Terbaik: 0.0005
Generasi 14/30 | Fitness Terbaik: 0.0005
Generasi 15/30 | Fitness Terbaik: 0.0005
Generasi 16/30 | Fitness Terbaik: 0.0005
Generasi 17/30 | Fitness Terbaik: 0.0005
Generasi 18/30 | Fitness Terbaik: 0.0005
Generasi 19/30 | Fitness Terbaik: 0.0005
Generasi 20/30 | Fitness Terbaik: 0.0005
Generasi 21/30 | Fitness Terbaik: 0.0005
Generasi 22/30 | Fitness Terbaik: 0.0005
Generasi 23/30 | Fitness Terbaik: 0.0005
Generasi 24/30 | Fitness Terbaik: 0.0005
Generasi 25/30 | Fitness 

In [5]:
print("\n--- Proses Evolusi Selesai ---")
print(f"Kromosom Terbaik Ditemukan: {best_overall_chromosome}")
print(f"Skor Fitness (Adjusted R-squared) Tertinggi: {best_overall_fitness:.4f}")

# Menerjemahkan kromosom terbaik menjadi nama fitur
selected_features = [feature_names[i] for i, gene in enumerate(best_overall_chromosome) if gene == 1]

print("\n✅ Kombinasi Faktor Paling Berpengaruh yang Ditemukan:")
for feature in selected_features:
    print(f"- {feature}")

print(f"\nInterpretasi: Kombinasi faktor di atas secara kolektif dapat menjelaskan sekitar **{best_overall_fitness*100:.2f}%** variasi dalam skor stres remaja berdasarkan data yang ada.")


--- Proses Evolusi Selesai ---
Kromosom Terbaik Ditemukan: [0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0]
Skor Fitness (Adjusted R-squared) Tertinggi: 0.0005

✅ Kombinasi Faktor Paling Berpengaruh yang Ditemukan:
- Sleep_Hours
- Support_System_Moderate
- Academic_Performance_Excellent

Interpretasi: Kombinasi faktor di atas secara kolektif dapat menjelaskan sekitar **0.05%** variasi dalam skor stres remaja berdasarkan data yang ada.
